# Experiments — *What Happens Next?* video classification

Lab notebook for tracking architectures, runs, and results on the CSC_43M04_EP challenge (33-class Something-Something-v2-style data, 45,132 train / 6,746 val videos, 4 frames per video on disk).

## Modular architecture

All experiments here use a 3-slot modular video model (`models/modular.py`):

```
video  (B, T, 3, H, W)
  └─ SpatialEncoder    (B, T, d)    # per-frame, independent
  └─ TemporalProcessor (B, d')      # mixes across T
  └─ Classifier        (B, K)       # final logits
```

Each slot is swappable. Current registries:

| Slot | Options |
|---|---|
| Spatial | `resnet` (r18/r34/r50, pretrained opt.), `vit` (ti_16, s_16, b_16, b_32, l_16) |
| Temporal | `mean_pool`, `lstm`, `transformer` |
| Classifier | `linear`, `mlp` |

A run is defined by a Hydra config under `src/configs/experiment/<name>.yaml` that picks one option per slot + hyper-params. `build_model` checks shape compatibility between slots at construction time.

## Experiment 1 — ViT-S/16 + Transformer + MLP, from scratch

First run of the modular pipeline. Everything trained from random init (no ImageNet weights). Purpose: establish a from-scratch floor for a ViT-based architecture on this data scale, and validate the modular pipeline end-to-end.

**Config:** `experiment=vit_transformer_mlp` (`src/configs/experiment/vit_transformer_mlp.yaml`)

### Components

| Slot | Module | Config |
|---|---|---|
| Spatial | `ViTEncoder(variant=vit_s_16, pretrained=False)` | patch 16, 12 blocks, 6 heads, hidden 384, mlp 1536, 224² input → CLS → (B, T, 384) |
| Temporal | `TransformerTemporal(in_dim=384, num_heads=8, num_layers=2, dropout=0.1, max_len=64)` | prepend learnable CLS, learnable pos-embed, pre-norm TF blocks, take CLS → (B, 384) |
| Classifier | `MLPClassifier(in_dim=384, hidden_dim=512, num_classes=33, dropout=0.5)` | Linear→GELU→Dropout→Linear |

### Parameter budget (approx.)

| Module | Params |
|---|---|
| ViT-S/16 | ~22.0 M |
| Temporal transformer (2 layers, d=384) | ~1.8 M |
| MLP head (384→512→33) | ~0.2 M |
| **Total** | **~24.0 M** |

### Training setup

- **Data:** `num_frames=4`, `val_ratio=0.2` (internal split from `train/`), ImageNet normalization off (trained from scratch).
- **Optim:** Adam, `lr=1e-4`, no weight decay, no scheduler.
- **Batch:** `batch_size=8`, FP32, `num_workers=2`.
- **Epochs:** 20 (target ≈ 5 h wall-clock).
- **Hardware:** RTX 2000 Ada, 16 GB, 70 W TGP.
- **Checkpoint:** `/Data/challenge_sb/best_model_vit_s_scratch.pt` (best-by-val-acc).
- **Log:** `/Data/challenge_sb/logs/vit_s_scratch_20260423_185848.log`.
- **Started:** 2026-04-23 18:58.

### Hypothesis

From-scratch ViT at this data scale (45k videos ≈ 1/7 ImageNet) is expected to plateau well below a pretrained-CNN baseline. Realistic ceiling: **15–25 % val top-1**. The run is primarily a floor measurement and a pipeline sanity check — not a leaderboard contender.

### Results

*To be filled in after the run completes.*

| Epoch | Train loss | Train acc | Val loss | Val acc | Notes |
|---|---|---|---|---|---|
| 1 |  |  |  |  |  |
| … |  |  |  |  |  |
| 20 |  |  |  |  |  |

- **Best val top-1:** _TBD_
- **At epoch:** _TBD_
- **Wall-clock per epoch:** _TBD_ min (total: _TBD_ h)
- **Peak VRAM:** _TBD_ GB
- **Test-set submission score (if submitted):** _TBD_

#### Observations

_TBD — what the learning curve looks like, where overfitting starts, whether train/val gap matches the from-scratch hypothesis, any obvious next lever._

## Experiment 2 — ResNet18 + Transformer + MLP, from scratch

Second run of the modular pipeline. A CNN replaces the ViT in the spatial slot, still trained from random init (no ImageNet weights). Purpose: measure the CNN vs. ViT gap at this data scale (45k videos) with everything else held constant.

**Config:** `experiment=resnet_transformer_mlp` (`src/configs/experiment/resnet_transformer_mlp.yaml`)

### Components

| Slot | Module | Config |
|---|---|---|
| Spatial | `ResNetEncoder(variant=resnet18, pretrained=False)` | 4 stages, 224² input, adaptive avg-pool → per-frame 512-d → (B, T, 512) |
| Temporal | `TransformerTemporal(in_dim=512, num_heads=8, num_layers=2, dropout=0.1, max_len=64)` | prepend learnable CLS, learnable pos-embed, pre-norm TF blocks (GELU, dim_ff=2048), take CLS → (B, 512) |
| Classifier | `MLPClassifier(in_dim=512, hidden_dim=512, num_classes=33, dropout=0.5)` | Linear(512→512) → GELU → Dropout(0.5) → Linear(512→33) |

### Parameter budget (exact)

| Module | Params |
|---|---|
| ResNet18 | 11.18 M |
| Temporal transformer (2 layers, d=512) | 6.34 M |
| MLP head (512→512→33) | 0.28 M |
| **Total** | **17.80 M** |

### Training setup

- **Data:** `num_frames=4`, `val_ratio=0.2` (internal split from `train/`), ImageNet normalization off (trained from scratch). Train filter applied: the 23 extra shifted-scheme class folders (138 mislabeled clip duplicates) are excluded via a whitelist against `val/` folder names → 44,993 samples.
- **Optim:** Adam, `lr=1e-4`, no weight decay, no scheduler.
- **Batch:** `batch_size=8`, FP32, `num_workers=2`.
- **Epochs:** 20.
- **Hardware:** RTX A5000, 24 GB, CUDA 12.8.
- **Checkpoint:** `/Data/challenge_sb/best_model_resnet_tf_mlp_scratch.pt` (best-by-val-acc).
- **Log:** `/Data/challenge_sb/logs/resnet_transformer_mlp_20260423_194250.log`.
- **Started:** 2026-04-23 19:43.

### Hypothesis

At 45k videos, a from-scratch ResNet18 should learn faster than a from-scratch ViT-S (CNN inductive bias dominates when pretraining is absent). Expected ceiling: **20–30 % val top-1**, likely a few points above the ViT-scratch run. Still below a pretrained baseline, but useful as an ablation anchor for the spatial slot.

### Results

*To be filled in after the run completes.*

| Epoch | Train loss | Train acc | Val loss | Val acc | Notes |
|---|---|---|---|---|---|
| 1 |  |  |  |  |  |
| … |  |  |  |  |  |
| 20 |  |  |  |  |  |

- **Best val top-1:** _TBD_
- **At epoch:** _TBD_
- **Wall-clock per epoch:** _TBD_ min (total: _TBD_ h)
- **Peak VRAM:** _TBD_ GB
- **Test-set submission score (if submitted):** _TBD_

#### Observations

_TBD — whether the CNN's inductive bias gives it a clear lead over the ViT-scratch curve, whether the gap widens or closes over 20 epochs, any overfitting signatures._